# Periodic but not circular: a 24-hour activity rhythm

This example illustrates the *periodic but not circular* case: the predictor (time of day) repeats every 24 h, while the response itself is not an angle. The quantity of interest is *activity*, a level bounded in $(0, 1)$: the periodicity belongs to the predictor, not to the observation space of the outcome (unlike a **circular** quantity such as a compass bearing, where 0° and 360° are the *same* direction and the data live on a ring). We model activity with a **Beta** likelihood and let both its mean and its precision vary sinusoidally across the 24 h day.

The statistical model is a Beta GLM with a sinusoidal design on time:

- a **mean** activity level `mu(t)` driven by a linear predictor of a constant plus sine and cosine of the 24 h day, passed through a sigmoid so it stays in (0, 1);
- a positive **precision** `phi(t)` from a second sinusoidal linear predictor passed through a softplus;
- observations `activity ~ Beta(mu * phi, (1 - mu) * phi)`.


## Setup

Select the R (`ir`) kernel and follow the [R setup](../getting-started-r.md)
page for the compiler and kernel prerequisites. Run the cells in order.
Notebook execution is local by design (tests/test_notebooks.py).


In [ ]:
required <- c("brms", "posterior", "circular", "ggplot2", "ragg", "IRdisplay", "patchwork")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  install.packages(missing, repos = "https://cloud.r-project.org")
}

if (!requireNamespace("cmdstanr", quietly = TRUE)) {
  install.packages("cmdstanr",
    repos = c("https://stan-dev.r-universe.dev", "https://cloud.r-project.org"))
}
cmdstan_ok <- tryCatch(dir.exists(as.character(cmdstanr::cmdstan_path())),
  error = function(e) FALSE)
if (!cmdstan_ok) cmdstanr::install_cmdstan()


In [ ]:
versions <- c(R = as.character(getRversion()),
  vapply(c("brms", "posterior", "circular", "ggplot2", "ragg", "IRdisplay", "patchwork", "cmdstanr"),
    function(p) as.character(packageVersion(p)), character(1)))
versions


In [ ]:
suppressPackageStartupMessages({
  library(brms)
  library(posterior)
  library(circular)
  library(ggplot2)
  library(patchwork)
})


In [ ]:
repo_root <- if (file.exists(file.path("..", "R", "unwrap_family.R"))) {
  ".."
} else if (file.exists(file.path("R", "unwrap_family.R"))) {
  "."
} else {
  stop("Run from notebooks_r/ or the repository root.")
}

source(file.path(repo_root, "R", "unwrap_family.R"))
source(file.path(repo_root, "R", "unwrap_functions.R"))


In [ ]:
FIG_DIR <- file.path(repo_root, "figures", "periodic_not_circular")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)
# Render headless PNGs with ragg; keep archival vector PDFs alongside them.

show_figure <- function(plot, name, width, height, res = 110) {
  file <- file.path(FIG_DIR, paste0(name, ".png"))
  ragg::agg_png(file, width = width, height = height, units = "in",
                res = res)
  print(plot)
  invisible(dev.off())
  IRdisplay::display_png(file = file)
  invisible(NULL)
}

theme_set(theme_minimal(base_size = 12))
save_figure <- function(plot, name, width = 8, height = 5) {
  ggsave(file.path(FIG_DIR, paste0(name, ".pdf")), plot, width = width, height = height)
  show_figure(plot, name, width, height)
}


In [ ]:
SEED <- 42
quick_run <- Sys.getenv("UNWRAP_R_QUICK") == "1"
CHAINS <- if (quick_run) 2L else 4L
ITER <- if (quick_run) 3000L else 4000L
WARMUP <- if (quick_run) 1500L else 2000L
# These are local execution budgets, not convergence claims. Hard diagnostic
# gates below apply to both budgets; increase sampling if they fail.


In [ ]:
wrap <- function(x) atan2(sin(x), cos(x))
circ_mean <- function(x) atan2(mean(sin(x)), mean(cos(x)))
resultant <- function(x) Mod(mean(exp(1i * x)))
softplus_stable <- function(x) pmax(x, 0) + log1p(exp(-abs(x)))
log1p_exp <- softplus_stable
A1_safe <- function(k) {
  z <- besselI(k, 1, expon.scaled = TRUE) / besselI(k, 0, expon.scaled = TRUE)
  if (!is.null(dim(k))) dim(z) <- dim(k)
  z
}
vm_logpdf <- function(y, mu, kappa) {
  kappa * (cos(y - mu) - 1) - log(2 * pi) - log(besselI(kappa, 0, expon.scaled = TRUE))
}
hdi <- function(x, prob = 0.95) {
  x <- sort(as.numeric(x))
  stopifnot(length(x) > 1L, all(is.finite(x)))
  gap <- min(length(x) - 1L, floor(prob * length(x)))
  ii <- seq_len(length(x) - gap)
  j <- ii[which.min(x[ii + gap] - x[ii])]
  c(low = x[j], high = x[j + gap])
}
interval <- function(x, angular = FALSE, prob = 0.95) {
  m <- if (angular) circ_mean(x) else mean(x)
  z <- if (angular) m + wrap(x - m) else x
  c(mean = m, hdi(z, prob))
}
vmu <- function(m) circular::circular(
    m, units = "radians", template = "none", modulo = "2pi")
vm_ctrl <- list(units = "radians", template = "none", modulo = "2pi")
draw_vm <- function(mu, kappa) {
  wrap(as.numeric(circular::rvonmises(1, mu = vmu(mu), kappa = kappa,
                                     control.circular = vm_ctrl)))
}
draw_matrix <- function(fit, parameters) {
  as.data.frame(posterior::as_draws_df(posterior::subset_draws(
    posterior::as_draws_array(fit), variable = parameters)))
}

diagnose_fit <- function(fit, angular = character(), parameters = NULL) {
  # Probe exactly the flags retained by this brms/backend combination.
  divergences <- tryCatch({
    samples <- fit$fit@sim$samples
    if (is.null(samples) || !length(samples)) {
      NA_real_
    } else {
      div_flags <- lapply(samples, function(ch) ch[["divergent"]])
      if (any(vapply(div_flags, is.null, logical(1)))) NA_real_
      else sum(unlist(div_flags))
    }
  }, error = function(e) NA_real_)
  if (is.na(divergences)) {
    cat("Divergences: not retained in this fit object (see R-hat/ESS below).\n")
  } else {
    cat("Divergences:", divergences, "\n")
    stopifnot(divergences == 0)
  }
  a <- posterior::as_draws_array(fit)
  if (is.null(parameters)) {
    parameters <- posterior::variables(a)
    parameters <- parameters[grepl("^(b_|sd_|r_|kmu_)", parameters)]
  }
  stopifnot(length(parameters) > 0L)
  summary_frame <- do.call(rbind, lapply(parameters, function(p) {
    x <- as.matrix(a[, , p])
    # as_draws_array keeps the draws_array class, whose [ method does not
    # drop the parameter slice, so x is retained as one long column
    # (draws x chains flattened); the rhat/ESS calls below are written
    # for that layout (as in the P1 notebooks).
    stopifnot(all(is.finite(x)))
    is_angle <- p %in% angular
    z <- if (is_angle) wrap(x - circ_mean(x)) else x
    data.frame(parameter = p,
      mean = if (is_angle) circ_mean(x) else mean(x),
      sd = if (is_angle) sqrt(-2 * log(min(1, resultant(x)))) else sd(as.numeric(x)),
      rhat = if (is_angle) Rhat_unwrap(x, method = "sine") else posterior::rhat(x),
      ess_bulk = posterior::ess_bulk(z), row.names = NULL)
  }))
  print(summary_frame)
  stopifnot(all(is.finite(summary_frame$rhat)), all(summary_frame$rhat < 1.01),
    all(is.finite(summary_frame$ess_bulk)), all(summary_frame$ess_bulk > 400))
  summary_frame
}
trace_plot <- function(fit, parameters, angular = character()) {
  d <- draw_matrix(fit, parameters)
  long <- do.call(rbind, lapply(parameters, function(p) {
    x <- d[[p]]
    if (p %in% angular) x <- wrap(x - circ_mean(x))
    data.frame(parameter = p, value = x, iteration = d[[".iteration"]],
               chain = factor(d[[".chain"]]))
  }))
  ggplot(long, aes(iteration, value, colour = chain, group = chain)) +
    geom_line(linewidth = 0.25, alpha = 0.6) +
    facet_wrap(~ parameter, scales = "free_y", ncol = 2) +
    labs(x = "retained iteration", y = "draw (angular traces centered)") +
    theme(legend.position = "bottom")
}


## Figure style

Figures use portable sans text and the shared minimal theme. Each figure
renders inline through the headless ragg PNG helper and saves a vector PDF
under `figures/periodic_not_circular/`. The generated directory is gitignored.


## 1. Simulate the 24-hour rhythm

1,000 samples of activity are drawn across a 24 h day from hidden sinusoidal mean and precision curves (fixed seed, so the result is reproducible): `mu(t)`, where activity is centered through the day, and `phi(t)`, how concentrated activity is around that center, each a constant plus a sine and a cosine of the 24 h day. This is the canonical "periodic" parameterization - an angular basis in time, not data on a circle.


In [ ]:
set.seed(42)
num_points <- 1000L
time <- seq(0, 24, length.out = num_points)
sin24 <- sin(2 * pi * time / 24)
cos24 <- cos(2 * pi * time / 24)
mu_alpha_true <- 0.5
mu_sin_true <- 2.0
mu_cos_true <- 1.5
phi_alpha_true <- 2.0
phi_sin_true <- 1.0
phi_cos_true <- 0.5
mu_true <- plogis(mu_alpha_true + mu_sin_true * sin24 + mu_cos_true * cos24)
phi_true <- softplus_stable(phi_alpha_true + phi_sin_true * sin24 + phi_cos_true * cos24)
activity <- rbeta(num_points, mu_true * phi_true, (1 - mu_true) * phi_true)
# The continuous Beta outcome is in (0,1); guard finite-precision draws
# that round to an endpoint. The margin must also survive brms' JSON
# serialization to Stan (~15 significant figures), which would
# round 1 - 2^-52 to 1 and make the Beta likelihood -inf for all
# draws (every initial value rejected, no fit).
activity <- pmin(1 - 1e-10, pmax(1e-10, activity))
# EQUIVALENCE NOTE: same seed, distribution, grid, and truths as Python;
# R's rbeta and NumPy's beta sampler do not generate bit-identical values.
data <- data.frame(time, sin24, cos24, activity)
stopifnot(nrow(data) == 1000L, all(activity > 0 & activity < 1))
head(data)


## 2. Fit the periodic model

Notebook execution is local by design (tests/test_notebooks.py).
The six Normal priors match the Python model exactly. `brms::Beta`
uses `phi` for precision. With its log precision link, the nonlinear
predictor `log(softplus(eta))` gives precisely `phi = softplus(eta)`.
A plain log-linear precision regression would change the model.
`0 + Intercept` preserves the uncentered coefficient priors.


In [ ]:
form <- bf(activity ~ 0 + Intercept + sin24 + cos24,
  nlf(phi ~ log(log1p_exp(eta))),
  eta ~ 0 + Intercept + sin24 + cos24,
  family = brms::Beta(link = "logit", link_phi = "log"))
priors <- (
  prior("normal(0, 1)", class = "b", coef = "Intercept") +
  prior("normal(2, 1)", class = "b", coef = "sin24") +
  prior("normal(2, 1)", class = "b", coef = "cos24") +
  prior("normal(1, 1)", class = "b", coef = "Intercept", nlpar = "eta") +
  prior("normal(1, 1)", class = "b", coef = "sin24", nlpar = "eta") +
  prior("normal(0, 1)", class = "b", coef = "cos24", nlpar = "eta")
)
ITER <- 2000L
WARMUP <- 1000L
# Execution budgets only; the diagnostic gate applies at either chain count.
stan_code <- make_stancode(form, data = data, prior = priors)
stopifnot(grepl("log1p_exp", stan_code, fixed = TRUE))
cat(stan_code)
set.seed(42)
fit <- brm(form, data = data, prior = priors, backend = "cmdstanr",
  chains = CHAINS, iter = ITER, warmup = WARMUP, cores = CHAINS,
  seed = 42, control = list(adapt_delta = 0.95))
parameters <- c("b_Intercept", "b_sin24", "b_cos24",
                "b_eta_Intercept", "b_eta_sin24", "b_eta_cos24")
summary_frame <- diagnose_fit(fit, parameters = parameters)


In [ ]:
p_trace <- trace_plot(fit, parameters)
save_figure(p_trace, "trace", 9, 6)


In [ ]:
# Explicit [grid x draws] linear predictors; no recycled outer callback.
d <- draw_matrix(fit, parameters)
G <- length(time)
lin_mu <- matrix(rep(d$b_Intercept, each = G), nrow = G) +
  outer(sin24, d$b_sin24) + outer(cos24, d$b_cos24)
lin_phi <- matrix(rep(d$b_eta_Intercept, each = G), nrow = G) +
  outer(sin24, d$b_eta_sin24) + outer(cos24, d$b_eta_cos24)
mu_draws <- plogis(lin_mu)
phi_draws <- softplus_stable(lin_phi)
stopifnot(identical(dim(mu_draws), c(G, nrow(d))), identical(dim(phi_draws), dim(mu_draws)))
curve_table <- function(draws, truth, panel) {
  bounds <- t(apply(draws, 1, hdi))
  data.frame(time, mean = rowMeans(draws), low = bounds[, 1], high = bounds[, 2], truth, panel)
}
latent <- rbind(curve_table(mu_draws, mu_true, "mean activity"),
                 curve_table(phi_draws, phi_true, "precision phi"))
p_latent <- ggplot(latent, aes(time, mean)) +
  geom_ribbon(aes(ymin = low, ymax = high), fill = "#E69F00", alpha = 0.2) +
  geom_line() + geom_line(aes(y = truth), linetype = "dashed", colour = "#0072B2") +
  facet_wrap(~ panel, scales = "free_y") +
  labs(x = "Time (hours)", y = "fitted mean and 95% HDI; dashed = simulated truth")
save_figure(p_latent, "periodic_beta_latent_curves", 9, 4)


## 3. Reading the figure

The next cell samples the posterior predictive draws of `activity`, summarizes them (median, 50%/90% highest-density intervals), and draws the figure.

- Dots are the simulated observations.
- The solid line is the posterior median of activity over the day.
- The shaded bands are the 90% (outer) and 50% (inner) highest-density intervals across posterior predictive samples.
- The band widens where the posterior is less certain about the mean or the precision.

The recovered curve should track the injected sinusoidal shape and, crucially, *not* be forced to agree across the 0/24 boundary as a plain circular fit would be — the wrap is a periodic continuity, not an angular equivalence. This contrasts with the bearing analysis in ``notebooks_r/zebra_escape.ipynb``, where that equivalence is the whole point.


In [ ]:
set.seed(42)
use <- seq_len(ncol(mu_draws))
if (quick_run) use <- use[seq_len(min(length(use), 1000L))]
yrep <- matrix(0, nrow = G, ncol = length(use))
for (j in seq_along(use)) {
  k <- use[j]
  yrep[, j] <- rbeta(G, mu_draws[, k] * phi_draws[, k],
                        (1 - mu_draws[, k]) * phi_draws[, k])
}
b50 <- t(apply(yrep, 1, hdi, prob = 0.5))
b90 <- t(apply(yrep, 1, hdi, prob = 0.9))
ppc <- data.frame(time, median = apply(yrep, 1, median),
  low50 = b50[, 1], high50 = b50[, 2], low90 = b90[, 1], high90 = b90[, 2])
p_ppc <- ggplot(ppc, aes(time, median)) +
  geom_ribbon(aes(ymin = low90, ymax = high90), fill = "#E69F00", alpha = 0.18) +
  geom_ribbon(aes(ymin = low50, ymax = high50), fill = "#E69F00", alpha = 0.35) +
  geom_point(data = data, aes(time, activity), inherit.aes = FALSE, size = 0.6, alpha = 0.35) +
  geom_line(linewidth = 0.6) + scale_x_continuous(breaks = c(0, 6, 12, 18, 24)) +
  labs(x = "Time (hours)", y = "Activity")
save_figure(p_ppc, "periodic_beta_fit", 7, 4)
